In [1]:
import torch

In [2]:
torch.cuda.is_available()

True

In [3]:
class NpuStress(torch.nn.Module):
    LAYER_COUNT = 1000

    def __init__(self):
        super(NpuStress, self).__init__()

        for i in range(self.LAYER_COUNT):
            self.__setattr__(f"linear{i}", torch.nn.Linear(1024, 1024))
            self.__setattr__(f"relu{i}", torch.nn.ReLU())

        self.final_linear = torch.nn.Linear(1024, 1024)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for _ in range(self.LAYER_COUNT):
            self.__getattr__(f"linear{_}")(x)
            self.__getattr__(f"relu{_}")(x)

        x = self.final_linear(x)
        return x

In [4]:
model = NpuStress()
model.eval()

NpuStress(
  (linear0): Linear(in_features=1024, out_features=1024, bias=True)
  (relu0): ReLU()
  (linear1): Linear(in_features=1024, out_features=1024, bias=True)
  (relu1): ReLU()
  (linear2): Linear(in_features=1024, out_features=1024, bias=True)
  (relu2): ReLU()
  (linear3): Linear(in_features=1024, out_features=1024, bias=True)
  (relu3): ReLU()
  (linear4): Linear(in_features=1024, out_features=1024, bias=True)
  (relu4): ReLU()
  (linear5): Linear(in_features=1024, out_features=1024, bias=True)
  (relu5): ReLU()
  (linear6): Linear(in_features=1024, out_features=1024, bias=True)
  (relu6): ReLU()
  (linear7): Linear(in_features=1024, out_features=1024, bias=True)
  (relu7): ReLU()
  (linear8): Linear(in_features=1024, out_features=1024, bias=True)
  (relu8): ReLU()
  (linear9): Linear(in_features=1024, out_features=1024, bias=True)
  (relu9): ReLU()
  (linear10): Linear(in_features=1024, out_features=1024, bias=True)
  (relu10): ReLU()
  (linear11): Linear(in_features=1024, ou

In [5]:
input_tensor = torch.randn(10000, 1024)

output = model(input_tensor)

output

tensor([[-0.2245, -0.2855,  0.8850,  ...,  0.1066,  0.1830,  0.2301],
        [-0.1491, -0.2493, -0.2236,  ...,  0.9841, -0.3693, -0.5274],
        [-0.3650, -0.0445, -0.0099,  ..., -0.0972, -0.3628, -0.8801],
        ...,
        [-0.5698, -0.9021,  0.2707,  ...,  1.1809,  0.2074, -0.1759],
        [-0.1769, -0.7154, -0.1663,  ..., -0.5562,  1.0006, -0.0519],
        [-0.2489, -0.5125,  0.0707,  ...,  0.0550, -0.4267, -0.4911]],
       grad_fn=<AddmmBackward0>)

In [ ]:
traced_model = torch.jit.trace(model, input_tensor)

In [ ]:
import qai_hub as hub
device = hub.Device("Dragonwing RB3 Gen 2 Vision Kit")

# Optimize model for the chosen device
compile_job = hub.submit_compile_job(
        model=traced_model,
        device=device,
        input_specs=dict(image=(10000, 1024)),
        options="--target_runtime tflite"
)
target_model = compile_job.get_target_model()

Uploading tmpxre2d7hx.pt part 1 of 4


100%|█████████▉| 1.00G/1.00G [09:11<00:00, 2.27MB/s]